In [51]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("VideoGamesGraph") \
    .master("local[*]") \
    .config("spark.jars.packages",
            "org.neo4j:neo4j-connector-apache-spark_2.13:5.3.10_for_spark_3") \
    .getOrCreate()

spark

In [52]:
df = spark.read.csv(
    "/opt/spark/work-dir/notebooks/labs/data/rec-amz-Video-Games/rec-amz-Video-Games.edges",
    sep=",",
    inferSchema=True
)

df.show(5)
df.printSchema()

+--------------+----------+---+----------+
|           _c0|       _c1|_c2|       _c3|
+--------------+----------+---+----------+
| AB9S9279OZ3QO|0078764343|5.0|1373155200|
|A24SSUT5CSW8BH|0078764343|5.0|1377302400|
| AK3V0HEBJMQ7J|0078764343|4.0|1372896000|
|A10BECPH7W8HM7|043933702X|5.0|1404950400|
|A2PRV9OULX1TWP|043933702X|5.0|1386115200|
+--------------+----------+---+----------+
only showing top 5 rows
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: double (nullable = true)
 |-- _c3: integer (nullable = true)



In [53]:
df = df.toDF("user", "item", "rating", "timestamp")

df.show(5)

+--------------+----------+------+----------+
|          user|      item|rating| timestamp|
+--------------+----------+------+----------+
| AB9S9279OZ3QO|0078764343|   5.0|1373155200|
|A24SSUT5CSW8BH|0078764343|   5.0|1377302400|
| AK3V0HEBJMQ7J|0078764343|   4.0|1372896000|
|A10BECPH7W8HM7|043933702X|   5.0|1404950400|
|A2PRV9OULX1TWP|043933702X|   5.0|1386115200|
+--------------+----------+------+----------+
only showing top 5 rows


In [54]:
from pyspark.sql.functions import col

vertices = df.select(col("user").alias("id")) \
    .union(df.select(col("item").alias("id"))) \
    .distinct()

edges = df.select(
    col("user").alias("src"),
    col("item").alias("dst")
)

vertices.show(5)
edges.show(5)

+--------------+
|            id|
+--------------+
| AFWPLXT2OD6H1|
|A17SPEC8D1SX85|
|A1JJSCLG40POB0|
| AFVGYJ34XP26A|
|A1D2G65W3QHBIQ|
+--------------+
only showing top 5 rows
+--------------+----------+
|           src|       dst|
+--------------+----------+
| AB9S9279OZ3QO|0078764343|
|A24SSUT5CSW8BH|0078764343|
| AK3V0HEBJMQ7J|0078764343|
|A10BECPH7W8HM7|043933702X|
|A2PRV9OULX1TWP|043933702X|
+--------------+----------+
only showing top 5 rows


In [55]:
from pyspark.sql.functions import count

pagerank = edges.groupBy("dst") \
    .agg(count("*").alias("rank"))

pagerank.orderBy("rank", ascending=False).show(10)

+----------+-----+
|       dst| rank|
+----------+-----+
|B00DJFIMW6|16221|
|B00BGA9WK2| 7561|
|B00FAX6XQC| 5713|
|B009KS4XRO| 5489|
|B002VBWIP6| 5190|
|B0055SWM08| 4638|
|B00CSR2J9I| 4510|
|B0015AARJI| 4468|
|B00178630A| 3522|
|B000FKBCX4| 3290|
+----------+-----+
only showing top 10 rows


In [56]:
degrees = edges.groupBy("src").count()

degrees.groupBy("count") \
    .count() \
    .orderBy("count") \
    .show()

+-----+------+
|count| count|
+-----+------+
|    1|636867|
|    2|105627|
|    3| 36355|
|    4| 16891|
|    5|  9426|
|    6|  5609|
|    7|  3575|
|    8|  2540|
|    9|  1820|
|   10|  1371|
|   11|  1045|
|   12|   808|
|   13|   669|
|   14|   523|
|   15|   459|
|   16|   365|
|   17|   279|
|   18|   262|
|   19|   215|
|   20|   180|
+-----+------+
only showing top 20 rows


In [57]:
top_users = edges.groupBy("src") \
    .count() \
    .orderBy("count", ascending=False)

top_users.show(10)

+--------------+-----+
|           src|count|
+--------------+-----+
|A3V6Z4RCDGRC44|  880|
|A3W4D8XOGLWUN5|  817|
| AJKWF4W7QD4NS|  797|
|A2QHS1ZCIQOL7E|  521|
|A2TCG2HV1VJP6V|  474|
|A29BQ6B90Y1R5F|  429|
| AFV2584U13XP3|  338|
|A20DZX38KRBIT8|  320|
| A74TA8X5YQ7NE|  267|
|A2582KMXLK2P06|  263|
+--------------+-----+
only showing top 10 rows


In [58]:
top_items = edges.groupBy("dst") \
    .count() \
    .orderBy("count", ascending=False)

top_items.show(10)

+----------+-----+
|       dst|count|
+----------+-----+
|B00DJFIMW6|16221|
|B00BGA9WK2| 7561|
|B00FAX6XQC| 5713|
|B009KS4XRO| 5489|
|B002VBWIP6| 5190|
|B0055SWM08| 4638|
|B00CSR2J9I| 4510|
|B0015AARJI| 4468|
|B00178630A| 3522|
|B000FKBCX4| 3290|
+----------+-----+
only showing top 10 rows


In [ ]:
vertices.write.format("org.neo4j.spark.DataSource") \
    .mode("Overwrite") \
    .option("url", "bolt://neo4j-iteso:7687") \
    .option("authentication.type", "basic") \
    .option("authentication.basic.username", "neo4j") \
    .option("authentication.basic.password", "neo4j123") \
    .option("labels", ":Node") \
    .option("node.keys", "id") \
    .save()

[Stage 25:>                                                       (0 + 10) / 10]

In [ ]:
edges.write.format("org.neo4j.spark.DataSource") \
    .mode("Overwrite") \
    .option("url", "bolt://neo4j-iteso:7687") \
    .option("authentication.type", "basic") \
    .option("authentication.basic.username", "neo4j") \
    .option("authentication.basic.password", "neo4j123") \
    .option("relationship", "INTERACTS") \
    .option("relationship.save.strategy", "keys") \
    .option("relationship.source.labels", ":Node") \
    .option("relationship.target.labels", ":Node") \
    .option("relationship.source.node.keys", "src:id") \
    .option("relationship.target.node.keys", "dst:id") \
    .save()

In [ ]:
spark.stop()